# 📘 Bài Giảng Thực Hành: K-Nearest Neighbors (KNN Classification)
**Mục tiêu bài giảng:**
1. Hiểu nguyên lý thuật toán học lười (Lazy Learning) và khoảng cách Minkowski/Euclidean.
2. Sử dụng phương pháp Elbow (Error Rate Curve) để chọn tham số $K$ tối ưu.
3. Trực quan hóa ranh giới phân lớp (Decision Boundary) phi tuyến trên cả Train và Test.

---
### 📊 Tập dữ liệu: `Social_Network_Ads.csv`


## 1. Nạp Dữ Liệu & Chuẩn Hóa StandardScaler


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

dataset = pd.read_csv('Social_Network_Ads.csv')
X = dataset.iloc[:, [2, 3]].values
y = dataset.iloc[:, 4].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)


## 2. Tìm $K$ Tối Ưu Bằng Phương Pháp Elbow


In [ ]:
error_rates = []
for k in range(1, 31):
    knn = KNeighborsClassifier(n_neighbors=k, metric='minkowski', p=2)
    knn.fit(X_train_scaled, y_train)
    pred_k = knn.predict(X_test_scaled)
    error_rates.append(np.mean(pred_k != y_test))

optimal_k = np.argmin(error_rates) + 1
print(f'Giá trị K tối ưu nhất trên Test Set: K = {optimal_k} (Tỷ lệ sai số = {error_rates[optimal_k-1]*100:.2f}%)')


📌 **Expected Output:**
`Giá trị K tối ưu nhất trên Test Set: K = 5`


## 3. Huấn luyện KNeighborsClassifier với $K=5$ và Trực quan hóa Decision Boundary


In [ ]:
classifier = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
classifier.fit(X_train_scaled, y_train)
y_pred = classifier.predict(X_test_scaled)
print(f'Accuracy (K=5): {accuracy_score(y_test, y_pred)*100:.2f}%')

# Vẽ Decision Boundary
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for idx, (X_set, y_set, title) in enumerate([(X_train_scaled, y_train, 'Train'), (X_test_scaled, y_test, 'Test')]):
    X1, X2 = np.meshgrid(np.arange(X_set[:,0].min()-1, X_set[:,0].max()+1, 0.02), np.arange(X_set[:,1].min()-1, X_set[:,1].max()+1, 0.02))
    Z = classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape)
    axes[idx].contourf(X1, X2, Z, alpha=0.3, cmap=ListedColormap(('salmon', 'lightgreen')))
    for i, j in enumerate(np.unique(y_set)):
        axes[idx].scatter(X_set[y_set==j, 0], X_set[y_set==j, 1], color=ListedColormap(('red', 'green'))(i), label=f'Class {j}')
    axes[idx].set_title(f'KNN (K=5) - {title} Set', fontweight='bold')
    axes[idx].set_xlabel('Age (Scaled)')
    axes[idx].set_ylabel('Salary (Scaled)')
    axes[idx].legend()
plt.tight_layout()
plt.show()


📌 **Expected Output:**
Accuracy đạt 95% và hiển thị 2 biểu đồ ranh giới quyết định phân chia mượt mà giữa vùng Mua và Không mua.
